### A system that can understand text semantically (by meaning, not just keywords) using machine learning embeddings and find the most relevant matches from a custom dataset — reviews, feedback, logs, or any other text.


In [1]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

texts = ["great product", "poor customer service", "reliable and durable"]
embeddings = model.encode(texts)

e:\ML\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\ML\.venv\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [2]:
import socket

s = socket.socket()
s.connect(('127.0.0.1', 6333))  # Make sure something is listening here

In [3]:
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams

In [4]:
client = QdrantClient(host="127.0.0.1", port=6333)

In [5]:
client.recreate_collection(
    collection_name="customer_reviews",
    vectors_config=VectorParams(size=384, distance="Cosine")
)

C:\Users\Durga\AppData\Local\Temp\ipykernel_22080\3664281288.py:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [6]:
points = [
    PointStruct(
        id=i,
        vector=embeddings[i].tolist(),  # convert NumPy array to Python list
        payload={"text": texts[i]}
    )
    for i in range(len(texts))
]

In [7]:
client.upsert(collection_name="customer_reviews", points=points)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [8]:
query = "trustworthy product"
query_vector = model.encode(query)

results = client.search(
    collection_name="customer_reviews",
    query_vector=query_vector,
    limit=3
)

for hit in results:
    print(hit.payload["text"], hit.score)

reliable and durable 0.57864606
great product 0.44516364
poor customer service 0.3172544


C:\Users\Durga\AppData\Local\Temp\ipykernel_22080\273011142.py:4: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = client.search(
